In [1]:
!pip install -q uv
!uv pip install --system \
  "transformers==5.0.0" accelerate bitsandbytes peft trl datasets \
  soundfile librosa mutagen openpyxl datacollective

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 76.2 MB/s eta 0:00:00:00:0100:01
Using Python 3.12.13 environment at: /usr
Resolved 106 packages in 2.41s                                       
Prepared 6 packages in 721ms                                             
Uninstalled 1 package in 3ms
Installed 6 packages in 11ms                                
 + bitsandbytes==0.50.1
 + datacollective==0.5.7
 + fox-progress-bar==0.1.3
 + mutagen==1.48.1
 - requests==2.32.4
 + requests==2.34.2
 + trl==1.10.0


In [3]:
import os
import csv
import glob
import json as _json
import random
import difflib

import numpy as np
import pandas as pd
import torch
from mutagen.mp3 import MP3

from huggingface_hub import login

WHISPER_REPO_ID = "amirsz8203/whisper-small-fa-finetuned"
LANGUAGE = "persian"
TASK = "transcribe"
SAMPLE_RATE = 16000
SEED = 42
N_NEW_CLIPS = 10000       # کلیپ جدید برای transcribe (قبل از فیلتر)
MIN_SIMILARITY = 0.4
TARGET_IDENTICAL = 4000
TARGET_SUBSTITUTION = 6000

CV_EXTRACT_DIR = "/tmp/common_voice_fa_extracted"
OLD_COMBINED_PATH = "/kaggle/input/datasets/amirsafarzadeh8203/raw-pairs-v280/raw_pairs_v280.csv"  

random.seed(SEED)
np.random.seed(SEED)

try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    MDC_API_KEY = UserSecretsClient().get_secret("MDC_API_KEY")
except Exception:
    HF_TOKEN = "خودتان جایگذاری کنید"
    MDC_API_KEY = "خودتان جایگذاری کنید"

os.environ["MDC_API_KEY"] = MDC_API_KEY
login(token=HF_TOKEN)

In [4]:
CV_DATASET_ID = "cmqinhw5100v8nr07gyg5gi4v"

from datacollective import download_dataset
import tarfile

cv_archive_path = str(download_dataset(CV_DATASET_ID))

if os.path.isfile(cv_archive_path):
    if not os.path.isdir(CV_EXTRACT_DIR) or not os.listdir(CV_EXTRACT_DIR):
        os.makedirs(CV_EXTRACT_DIR, exist_ok=True)
        print("در حال extract کردن... (چند دقیقه طول می‌کشه)")
        with tarfile.open(cv_archive_path, "r:gz") as tar:
            tar.extractall(path=CV_EXTRACT_DIR)
    cv_root = CV_EXTRACT_DIR
elif os.path.isdir(cv_archive_path):
    cv_root = cv_archive_path
else:
    raise RuntimeError(f"مسیر برگشتی نه فایله نه پوشه: {cv_archive_path}")

tsv_candidates = (
    glob.glob(os.path.join(cv_root, "**", "validated.tsv"), recursive=True)
    or glob.glob(os.path.join(cv_root, "**", "train.tsv"), recursive=True)
    or glob.glob(os.path.join(cv_root, "**", "*.tsv"), recursive=True)
)
cv_tsv_path = tsv_candidates[0]
mp3_candidates = glob.glob(os.path.join(cv_root, "**", "*.mp3"), recursive=True)
clips_dir = os.path.dirname(mp3_candidates[0])

cv_df = pd.read_csv(cv_tsv_path, sep="\t", quoting=csv.QUOTE_NONE)
cv_df["full_path"] = cv_df["path"].apply(lambda p: os.path.join(clips_dir, p))
cv_df = cv_df.sample(frac=1.0, random_state=SEED).reset_index(drop=True)


def mp3_duration_sec(path):
    try:
        return MP3(path).info.length
    except Exception:
        return None


print("تعداد کل ردیف‌های متادیتا:", len(cv_df))

█████████████████████████████████████████████████🦊 100.0% (10.5 GB/10.5 GB) Average: 51.7 MB/s Total time: 03:27
در حال extract کردن... (چند دقیقه طول می‌کشه)


/tmp/ipykernel_58/3422193343.py:13: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(path=CV_EXTRACT_DIR)
/tmp/ipykernel_58/3422193343.py:29: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  cv_df = pd.read_csv(cv_tsv_path, sep="\t", quoting=csv.QUOTE_NONE)


تعداد کل ردیف‌های متادیتا: 341657


In [5]:
ROUND_CONFIGS = [
    {"round": 1, "hours": 20, "replay_ratio": 0.0, "replay_hours_cap": None},
    {"round": 2, "hours": 15, "replay_ratio": 0.15, "replay_hours_cap": None},
    {"round": 3, "hours": 15, "replay_ratio": 0.10, "replay_hours_cap": None},
    {"round": 4, "hours": 15, "replay_ratio": 0.10, "replay_hours_cap": None},
    {"round": 5, "hours": 15, "replay_ratio": None, "replay_hours_cap": 5},
]


def select_round_full(cv_df, previous_clips, cfg):
    previous_clips = list(previous_clips)
    replay_paths = set()
    if cfg["replay_hours_cap"] is not None:
        shuffled_prev = previous_clips.copy()
        random.Random(SEED).shuffle(shuffled_prev)
        acc = 0.0
        target = cfg["replay_hours_cap"] * 3600
        for p in shuffled_prev:
            if acc >= target:
                break
            dur = mp3_duration_sec(p)
            if dur is None:
                continue
            replay_paths.add(p)
            acc += dur
    elif cfg["replay_ratio"]:
        n_replay = int(len(previous_clips) * cfg["replay_ratio"])
        if n_replay > 0:
            replay_paths = set(random.Random(SEED).sample(previous_clips, n_replay))

    used_paths_exclude = set(previous_clips) - replay_paths

    selected = []
    replay_selected = []
    total_sec = 0.0
    target_sec = cfg["hours"] * 3600

    for _, row in cv_df.iterrows():
        path = row["full_path"]
        sentence = str(row["sentence"])
        if len(sentence) > 400:
            continue
        if path in replay_paths:
            replay_selected.append({"path": path, "sentence": sentence})
            continue
        if path in used_paths_exclude:
            continue
        dur = mp3_duration_sec(path)
        if dur is None:
            continue
        selected.append({"path": path, "sentence": sentence})
        total_sec += dur
        if total_sec >= target_sec:
            break

    records = selected + replay_selected
    random.Random(SEED).shuffle(records)

    full_df = pd.DataFrame(records)
    n_eval = max(1, int(len(full_df) * 0.1))
    shuffled_idx = list(full_df.index)
    random.Random(SEED).shuffle(shuffled_idx)
    eval_idx = shuffled_idx[:n_eval]
    train_idx = shuffled_idx[n_eval:]

    train_df = full_df.loc[train_idx].reset_index(drop=True)
    eval_df = full_df.loc[eval_idx].reset_index(drop=True)

    new_used = set(previous_clips) | {r["path"] for r in selected}
    return train_df, eval_df, new_used


previous_clips = set()
all_train_records = []

for cfg in ROUND_CONFIGS:
    train_df, eval_df, previous_clips = select_round_full(cv_df, previous_clips, cfg)
    print(f"راند {cfg['round']}: train شامل {len(train_df)} کلیپ")
    for _, row in train_df.iterrows():
        all_train_records.append({"round": cfg["round"], "path": row["path"], "sentence": row["sentence"]})

train_pool_df = pd.DataFrame(all_train_records).drop_duplicates(subset=["path"]).reset_index(drop=True)
print("مجموع کلیپ‌های train (بدون تکرار):", len(train_pool_df))

راند 1: train شامل 16495 کلیپ
راند 2: train شامل 14795 کلیپ
راند 3: train شامل 15210 کلیپ
راند 4: train شامل 16491 کلیپ
راند 5: train شامل 16470 کلیپ
مجموع کلیپ‌های train (بدون تکرار): 67144


In [7]:
old_combined_df = pd.read_csv(OLD_COMBINED_PATH)
already_used_paths = set(old_combined_df["path"])
print("کلیپ‌هایی که قبلاً توی دیتاست هستن:", len(already_used_paths))

fresh_train_pool_df = train_pool_df[~train_pool_df["path"].isin(already_used_paths)].reset_index(drop=True)
print("کلیپ‌های تازه‌ی باقی‌مانده در train pool:", len(fresh_train_pool_df))

new_sample_df = fresh_train_pool_df.sample(
    n=min(N_NEW_CLIPS, len(fresh_train_pool_df)), random_state=SEED
).reset_index(drop=True)
print("نمونه‌ی جدید انتخاب‌شده برای transcribe:", len(new_sample_df))

کلیپ‌هایی که قبلاً توی دیتاست هستن: 5521
کلیپ‌های تازه‌ی باقی‌مانده در train pool: 65053
نمونه‌ی جدید انتخاب‌شده برای transcribe: 10000


In [9]:
from transformers import WhisperProcessor, WhisperForConditionalGeneration
import soundfile as sf
import librosa

processor = WhisperProcessor.from_pretrained(WHISPER_REPO_ID, language=LANGUAGE, task=TASK)
whisper_model = WhisperForConditionalGeneration.from_pretrained(WHISPER_REPO_ID)
whisper_model.generation_config.language = LANGUAGE
whisper_model.generation_config.task = TASK
whisper_model.generation_config.forced_decoder_ids = None
if torch.cuda.is_available():
    whisper_model = whisper_model.to("cuda:0")
whisper_model.eval()


def transcribe(path):
    audio, sr = sf.read(path)
    if audio.ndim > 1:
        audio = audio.mean(axis=1)
    if sr != SAMPLE_RATE:
        audio = librosa.resample(audio, orig_sr=sr, target_sr=SAMPLE_RATE)
    input_features = processor.feature_extractor(audio, sampling_rate=SAMPLE_RATE).input_features[0]
    input_features = torch.tensor(input_features).unsqueeze(0)
    if torch.cuda.is_available():
        input_features = input_features.to(whisper_model.device)
    with torch.no_grad():
        pred_ids = whisper_model.generate(input_features)
    return processor.batch_decode(pred_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0].strip()


raw_texts = []
for i, row in new_sample_df.iterrows():
    raw_texts.append(transcribe(row["path"]))
    if (i + 1) % 1000 == 0:
        print(f"{i + 1}/{len(new_sample_df)} transcribe شد")

new_sample_df["raw_whisper"] = raw_texts

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

1000/10000 transcribe شد
2000/10000 transcribe شد
3000/10000 transcribe شد
4000/10000 transcribe شد
5000/10000 transcribe شد
6000/10000 transcribe شد
7000/10000 transcribe شد
8000/10000 transcribe شد
9000/10000 transcribe شد
10000/10000 transcribe شد


In [ ]:
import transformers, huggingface_hub, requests
print(transformers.__version__, huggingface_hub.__version__, requests.__version__)

In [10]:
def word_similarity(raw, clean):
    matcher = difflib.SequenceMatcher(None, raw.split(), clean.split())
    return matcher.ratio()


new_sample_df["word_count_match"] = new_sample_df.apply(
    lambda r: len(str(r["raw_whisper"]).split()) == len(str(r["sentence"]).split()), axis=1
)
new_sample_df["similarity"] = new_sample_df.apply(
    lambda r: word_similarity(str(r["raw_whisper"]), str(r["sentence"])), axis=1
)

new_valid_df = new_sample_df[
    new_sample_df["word_count_match"] & (new_sample_df["similarity"] >= MIN_SIMILARITY)
].reset_index(drop=True)

new_identical_df = new_valid_df[
    new_valid_df["raw_whisper"].astype(str) == new_valid_df["sentence"].astype(str)
].reset_index(drop=True)
new_substitution_df = new_valid_df[
    new_valid_df["raw_whisper"].astype(str) != new_valid_df["sentence"].astype(str)
].reset_index(drop=True)

print(f"نمونه‌های جدید معتبر: {len(new_valid_df)} از {len(new_sample_df)}")
print(f"  identical جدید: {len(new_identical_df)}")
print(f"  substitution جدید: {len(new_substitution_df)}")

نمونه‌های جدید معتبر: 7552 از 10000
  identical جدید: 5106
  substitution جدید: 2446


In [11]:
old_identical_df = old_combined_df[
    old_combined_df["raw_whisper"].astype(str) == old_combined_df["sentence"].astype(str)
].reset_index(drop=True)
old_substitution_df = old_combined_df[
    old_combined_df["raw_whisper"].astype(str) != old_combined_df["sentence"].astype(str)
].reset_index(drop=True)

print(f"identical قدیمی: {len(old_identical_df)}  |  substitution قدیمی: {len(old_substitution_df)}")

all_identical_df = pd.concat([old_identical_df, new_identical_df], ignore_index=True).drop_duplicates(subset=["path"])
all_substitution_df = pd.concat([old_substitution_df, new_substitution_df], ignore_index=True).drop_duplicates(subset=["path"])

print(f"مجموع identical در دسترس: {len(all_identical_df)}")
print(f"مجموع substitution در دسترس: {len(all_substitution_df)}")

final_identical_df = all_identical_df.sample(
    n=min(TARGET_IDENTICAL, len(all_identical_df)), random_state=SEED
).reset_index(drop=True)
final_substitution_df = all_substitution_df.sample(
    n=min(TARGET_SUBSTITUTION, len(all_substitution_df)), random_state=SEED
).reset_index(drop=True)

final_combined_df = pd.concat([final_identical_df, final_substitution_df], ignore_index=True)
final_combined_df = final_combined_df.sample(frac=1.0, random_state=SEED).reset_index(drop=True)

print(f"\nنتیجه‌ی نهایی: {len(final_combined_df)} نمونه")
print(f"  identical: {len(final_identical_df)} ({100 * len(final_identical_df) / len(final_combined_df):.1f}%)")
print(f"  substitution: {len(final_substitution_df)} ({100 * len(final_substitution_df) / len(final_combined_df):.1f}%)")

identical قدیمی: 2208  |  substitution قدیمی: 3313
مجموع identical در دسترس: 7314
مجموع substitution در دسترس: 5759

نتیجه‌ی نهایی: 9759 نمونه
  identical: 4000 (41.0%)
  substitution: 5759 (59.0%)


In [ ]:
final_out_path = "/kaggle/working/raw_pairs_v10k.csv"
final_combined_df[["path", "sentence", "raw_whisper"]].to_csv(final_out_path, index=False)
print("دیتاست ترکیبی نهایی ذخیره شد:", final_out_path)